# 3.1 Position Encoding

> Absolute VS Relative

- What is absolute PE:
    - Fuses positional information into *input embeddings* via absolute positions in the whole sequence.
    - Examples: Sinusoidal, RoPE
- What is relative PE:
    - Provides relative distance information by impacting *attention scores*.
    - Examples: ALiBi

> How to judge absolute/relative

**Translation Invariance**: If after translating the positions, the attention score won't change, then it's relative encoding. (Mathmatical Property)

> The advantages, when to use relative/absolute

Relative PE is more suitable for long sequence generation.

RoPE is the main-stream choice nowadays. For that it's in the form of absolute PE but capable of relative PE mathmatical outputs.

In [2]:
import torch.nn as nn
import torch

## Sinusoidal
This positional encoding is directly added to input embeddings.

$$sinusoidal(i,2k)=sin(\frac{i}{\theta^{2k/d}}), \theta=10000$$

$$sinusoidal(i,2k+1) = cos(\frac{i}{\theta^{2k/d}}) $$

$i$ is the token position in the sequence, $k \in {1,2,...,d/2-1}$ is the dimention.

In [ ]:
class Sinusoidal(nn.Module):
    def __init__(self, d_model, max_len=5000, theta=10000.0):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        # use log to avoid overflow
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(theta)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # size of x: [seq_len, d_model]
        x = x + self.pe[:x.size(0), :]
        return x

## ALiBi

![](../figs/3.1-Positional-Encoding.png)

*This picture is from Train Short, Test Long: Attention with Linear Biases Enables Input Length Extrapolation [Press et al., 2021]*

In [ ]:
class ALiBi(nn.Module):
    def __init__(self, num_heads, max_seq_len, m=1.0):
        super().__init__()
        self.num_heads = num_heads
        self.max_seq_len = max_seq_len
        self.alibi = self._create_alibi()
        self.m = m

    def _create_alibi(self):
        # Create the ALiBi tensor based on the number of heads and max sequence length
        alibi = torch.zeros((self.num_heads, self.max_seq_len, self.max_seq_len))
        for head in range(self.num_heads):
            for i in range(self.max_seq_len):
                for j in range(self.max_seq_len):
                    if j > i:
                        alibi[head, i, j] = (j - i) * (head + 1) * self.m
        return alibi

    def forward(self, x):
        # Apply ALiBi to the input tensor x
        batch_size, seq_len, _ = x.size()
        alibi_slice = self.alibi[:, :seq_len, :seq_len]
        return x + alibi_slice.unsqueeze(0)  # Broadcasting to match batch size

## RoPE

In attention, Q and K will be multiplied by RoPE matrix.
$Rope(q_i) = R_iW_qx_i, Rope(k_i) = R_iW_kx_i$

$$ \theta_{i,k} = \frac{i}{\Theta^{(2k-2)/d}}, R_k^i = 
\begin{bmatrix}
\cos(\theta_{i,k}) & -\sin(\theta_{i,k}) \\
\sin(\theta_{i,k}) & \cos(\theta_{i,k})
\end{bmatrix}, k \in \{1,2,...,d/2\}
$$

$$ R^i = 
\begin{bmatrix}
R_1^i & 0 & 0 & \cdots & 0 \\
0 & R_2^i & 0 & \cdots & 0 \\
0 & 0 & R_3^i & \cdots & 0 \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
0 & 0 & 0 & \cdots & R_{d/2}^i
\end{bmatrix}
$$

So, for any input of size $(...,d,1)$, the multiply result will be:

$$
R^iq^i = 
\begin{bmatrix}
cos(\theta_{i,1})q_{i,1}-sin(\theta_{i,1})q_{i,2} \\
cos(\theta_{i,1})q_{i,2}+sin(\theta_{i,1})q_{i,1} \\
\dots \\
cos(\theta_{i,d/2})q_{i,d-1}-sin(\theta_{i,d/2})q_{i,d} \\
cos(\theta_{i,d/2})q_{i,d}+sin(\theta_{i,d/2})q_{i,d-1}
\end{bmatrix}
$$

Thus the RoPE can be computed as two addable objects like the following:

Repeated $cos$ and $sin$ like $[cos(\theta_{i,1}), cos(\theta_{i,1}), ... , cos(\theta_{i,d/2}), cos(\theta_{i,d/2})]$

dot product

original $x_i$ and $[-x_{i,2}, x_{i,1}, -x_{i,4}, x_{i,3}, ..., -x_{i,d-1}, x_{i,d}]$

In [7]:
x = torch.Tensor([1,2,3,4,5,6,7,8,9,10])

print(x[1::2])

sin_x = torch.stack([-x[1::2], x[::2]], dim=-1).view(-1)

print(sin_x)

tensor([ 2.,  4.,  6.,  8., 10.])
tensor([ -2.,   1.,  -4.,   3.,  -6.,   5.,  -8.,   7., -10.,   9.])


In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, theta: float, d_k: int, max_seq_len: int, 
                 device: torch.device=None, dtype: torch.dtype=None):
        super().__init__()
        self.dim = d_k
        self.theta = theta
        
        self.max_seq_len = max_seq_len
        i = torch.arange(1, d_k // 2 + 1, dtype=dtype)
        freq = 1.0 / (theta ** ((2*i-2) / self.dim))
        position = torch.arange(max_seq_len, dtype=dtype)
        angles = position.unsqueeze(1) * freq
        self.cos_cache = torch.cos(angles) # max_seq_len * d/2
        self.sin_cache = torch.sin(angles)
        # self.register_buffer(name="cos_cache", tensor=self.cos_cache, persistent=False)
        # self.register_buffer(name="sin_cache", tensor=self.sin_cache, persistent=False)

    def forward(self, x: torch.Tensor, token_positions: torch.Tensor) -> torch.Tensor:
        # shape of x: bs, num_heads, seq_len, d
        cos_emb = torch.repeat_interleave(self.cos_cache[token_positions], repeats=2, dim=-1).unsqueeze(0).unsqueeze(0)
        sin_emb = torch.repeat_interleave(self.sin_cache[token_positions], repeats=2, dim=-1).unsqueeze(0).unsqueeze(0)

        sin_x = torch.stack([-x[...,1::2], x[...,::2]], dim=-1).view(x.shape)
        return (x*cos_emb + sin_x * sin_emb).squeeze(0)

### RoPE scaling
